# Aggregate Multi-Seed Results — v2 Pipeline

Pipeline: `train.py` → `decoder_mod_v2.py` → `imputation_v2.py` → `classification.py`

Supports two experiment layouts:

1. **Main experiment** (`slurm_main_experiment.sh`) — single stage, 5 seeds:
   `experiments/results/Experiment_sd{seed}/`

2. **Ablation study** (`slurm_abstudy.sh`) — multiple stages, 5 seeds:
   `experiments/results/{stage}_sd{seed}/`

**Outputs** (written to `OUTPUT_DIR`):
- `classification_aggregated.csv`
- `imputation_aggregated.csv` (only when `imputation_v2_all_modalities.csv` exists)

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

## Configuration

Edit the parameters below before running. Set `EXPERIMENT_TYPE` to `"main"` or `"ablation"`, or set `STAGES` manually to override.

In [2]:
# --- Paths ---
RESULTS_DIR = "results_ablation"   # Root directory containing {stage}_sd{seed}/ subdirectories
OUTPUT_DIR  = "results_ablation_agg"   # Where to write aggregated CSVs

# --- Experiment type ---
EXPERIMENT_TYPE = "ablation"

# --- Seeds ---
SEEDS = [1, 2, 3, 4, 42]

# --- Stages ---
STAGES = [
    "S_trim_nofilter",
    "S_trim_topk1",
    "S_trim_topk3",
    "S_trim_topk5",
    "S_trim_topk10",
    "S_trim_topk20",
    "S_trim_topk1_gcn1_rr1",
    "S_trim_topk3_gcn1_rr1",
    "S_trim_topk5_gcn1_rr1",
    "S_trim_topk10_gcn1_rr1",
    "S_trim_topk20_gcn1_rr1",
    "S_trim_nofilter_gcn1_rr1",
    "S_trim_topk1_rr1_ma1",
    "S_trim_topk3_rr1_ma1",
    "S_trim_topk5_rr1_ma1",
    "S_trim_topk10_rr1_ma1",
    "S_trim_topk20_rr1_ma1",
    "S_trim_nofilter_rr1_ma1",
]

In [3]:
# --- Constants (do not normally need to change) ---

MAIN_STAGES = [""]

ABLATION_STAGES = [
    "S_trim_nofilter",
    "S_trim_topk1",
    "S_trim_topk3",
    "S_trim_topk5",
    "S_trim_topk10",
    "S_trim_topk20",
    "S_trim_topk1_gcn1_rr1",
    "S_trim_topk3_gcn1_rr1",
    "S_trim_topk5_gcn1_rr1",
    "S_trim_topk10_gcn1_rr1",
    "S_trim_topk20_gcn1_rr1",
    "S_trim_nofilter_gcn1_rr1",
    "S_trim_topk1_rr1_ma1",
    "S_trim_topk3_rr1_ma1",
    "S_trim_topk5_rr1_ma1",
    "S_trim_topk10_rr1_ma1",
    "S_trim_topk20_rr1_ma1",
    "S_trim_nofilter_rr1_ma1",
]

CLF_FILE = "output/classification_all_methods.csv"
IMP_FILE = "output/imputation_all_modalities.csv"

# Canonical stage name → old name used in error_abstudy_experiment
STAGE_NAME_MAP = {
}

# Sources for already-aggregated per-stage CSVs (priority order)
ABSTUDY_AGG_SOURCES = [
    {"base": "results_ablation_agg",       "stage_map": {}},
]

## Helper Functions

In [4]:
def resolve_stages(experiment_type, stages):
    if stages:
        return stages
    if experiment_type == "ablation":
        return ABLATION_STAGES
    return MAIN_STAGES


def compute_ci(values):
    """Return (mean, ci_lo, ci_hi, n) using 95% Student-t CI."""
    arr = np.array([v for v in values if np.isfinite(v)], dtype=float)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = arr.mean()
    if n == 1:
        return mean, np.nan, np.nan, 1
    lo, hi = stats.t.interval(0.95, df=n - 1, loc=mean, scale=stats.sem(arr))
    return mean, lo, hi, n


def run_dirs(results_dir, stages, seeds):
    """Yield (stage, seed, dir_path) for every (stage, seed) combination.
    Ablation layout: {results_dir}/{stage}/seed_{seed}/
    """
    for stage in stages:
        for seed in seeds:
            yield stage, seed, os.path.join(results_dir, stage, f"seed_{seed}")

## Resolve Configuration

In [5]:
stages     = resolve_stages(EXPERIMENT_TYPE, STAGES)
seeds      = SEEDS
results_dir = RESULTS_DIR
output_dir  = OUTPUT_DIR or RESULTS_DIR
os.makedirs(output_dir, exist_ok=True)

exp_type = EXPERIMENT_TYPE or ("custom" if STAGES else "main")
print(f"Experiment type : {exp_type}")
print(f"Stages          : {stages}")
print(f"Seeds           : {seeds}")
print(f"Results dir     : {results_dir}")
print(f"Output dir      : {output_dir}")

Experiment type : ablation
Stages          : ['S_trim_nofilter', 'S_trim_topk1', 'S_trim_topk3', 'S_trim_topk5', 'S_trim_topk10', 'S_trim_topk20', 'S_trim_topk1_gcn1_rr1', 'S_trim_topk3_gcn1_rr1', 'S_trim_topk5_gcn1_rr1', 'S_trim_topk10_gcn1_rr1', 'S_trim_topk20_gcn1_rr1', 'S_trim_nofilter_gcn1_rr1', 'S_trim_topk1_rr1_ma1', 'S_trim_topk3_rr1_ma1', 'S_trim_topk5_rr1_ma1', 'S_trim_topk10_rr1_ma1', 'S_trim_topk20_rr1_ma1', 'S_trim_nofilter_rr1_ma1']
Seeds           : [1, 2, 3, 4, 42]
Results dir     : results_ablation
Output dir      : results_ablation_agg


## Completeness Check

In [6]:
print("Seed completeness check:")
for stage in stages:
    print(f"  Stage: {stage}")
    for seed in seeds:
        d = os.path.join(results_dir, f"{stage}_sd{seed}")
        markers = {
            "train":      os.path.join(d, "output", "best_model.pth"),
            "decoder_v2": os.path.join(d, "output", "best_model_feat_finetuned.pth"),
            "imp_v2":     os.path.join(d, IMP_FILE),
            "classify":   os.path.join(d, CLF_FILE),
        }
        status = {k: ("✓" if os.path.exists(v) else "✗") for k, v in markers.items()}
        line = "  ".join(f"{k}:{s}" for k, s in status.items())
        print(f"    seed {seed:>3}: {line}")

Seed completeness check:
  Stage: S_trim_nofilter
    seed   1: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   2: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   3: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   4: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed  42: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
  Stage: S_trim_topk1
    seed   1: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   2: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   3: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   4: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed  42: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
  Stage: S_trim_topk3
    seed   1: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   2: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   3: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed   4: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
    seed  42: train:✗  decoder_v2:✗  imp_v2:✗  classify:✗
  Stage: S_trim_topk5
    seed   1: 

## Aggregate Classification Results

In [ ]:
def aggregate_classification(results_dir, stages, seeds, output_dir):
    all_dfs = []
    for stage, seed, d in run_dirs(results_dir, stages, seeds):
        path = os.path.join(d, CLF_FILE)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["stage"] = stage
            df["seed"] = seed
            all_dfs.append(df)
        else:
            print(f"  [warn] missing: {path}")

    if not all_dfs:
        print("  No classification CSVs found — skipping.")
        return None

    combined = pd.concat(all_dfs, ignore_index=True)
    metrics = ["Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro", "Precision_Weighted", "Recall_Weighted", "F1_Weighted", "AUROC", "AUPRC"]
    rows = []

    for (stage, imp, clf), grp in combined.groupby(["stage", "ImpMethod", "Classifier"]):
        row = {"Stage": stage, "ImpMethod": imp, "Classifier": clf, "n_seeds": grp["seed"].nunique()}
        for m in metrics:
            if m not in grp.columns:
                continue
            mean, lo, hi, n = compute_ci(grp[m].values)
            row[f"{m}_mean"]  = round(mean, 4) if np.isfinite(mean) else np.nan
            row[f"{m}_ci_lo"] = round(lo,   4) if np.isfinite(lo)   else np.nan
            row[f"{m}_ci_hi"] = round(hi,   4) if np.isfinite(hi)   else np.nan
            row[f"{m}_std"]   = round(float(np.std(grp[m].dropna())), 4)
        rows.append(row)

    out_df = pd.DataFrame(rows).sort_values(["Stage", "AUROC_mean"], ascending=[True, False])

    # Save per-stage files: {stage}_classification_aggregated.csv
    os.makedirs(output_dir, exist_ok=True)
    for stage in stages:
        stage_df = out_df[out_df["Stage"] == stage]
        if stage_df.empty:
            continue
        out_path = os.path.join(output_dir, f"{stage}_classification_aggregated.csv")
        stage_df.to_csv(out_path, index=False)
        print(f"  classification [{stage}] -> {out_path}")

    return out_df


print("Aggregating classification results...")
clf_df = aggregate_classification(results_dir, stages, seeds, output_dir)

Aggregating classification results...
  [warn] missing: results_ablation/S_trim_topk3_rr1_ma1/seed_1/output/classification_all_methods.csv
  [warn] missing: results_ablation/S_trim_topk3_rr1_ma1/seed_2/output/classification_all_methods.csv
  [warn] missing: results_ablation/S_trim_topk3_rr1_ma1/seed_3/output/classification_all_methods.csv
  [warn] missing: results_ablation/S_trim_topk3_rr1_ma1/seed_4/output/classification_all_methods.csv
  [warn] missing: results_ablation/S_trim_topk3_rr1_ma1/seed_42/output/classification_all_methods.csv
  classification [S_trim_nofilter] -> results_ablation_agg/S_trim_nofilter_classification_aggregated.csv
  classification [S_trim_topk1] -> results_ablation_agg/S_trim_topk1_classification_aggregated.csv
  classification [S_trim_topk3] -> results_ablation_agg/S_trim_topk3_classification_aggregated.csv
  classification [S_trim_topk5] -> results_ablation_agg/S_trim_topk5_classification_aggregated.csv
  classification [S_trim_topk10] -> results_ablation_a

In [8]:
# Classification summary table
if clf_df is not None:
    print("Classification Summary (sorted by Stage, then AUROC):")
    print(f"  {'Stage':<35} {'ImpMethod':<22} {'Classifier':<22} {'AUROC':>8} {'95% CI':>18} {'AUPRC':>8}")
    print("  " + "-" * 117)
    for _, r in clf_df.iterrows():
        ci_str = f"[{r.get('AUROC_ci_lo', np.nan):.4f}, {r.get('AUROC_ci_hi', np.nan):.4f}]"
        print(f"  {r['Stage']:<35} {r['ImpMethod']:<22} {r['Classifier']:<22} "
              f"{r['AUROC_mean']:>8.4f} {ci_str:>18} {r.get('AUPRC_mean', np.nan):>8.4f}")

Classification Summary (sorted by Stage, then AUROC):
  Stage                               ImpMethod              Classifier                AUROC             95% CI    AUPRC
  ---------------------------------------------------------------------------------------------------------------------
  S_trim_nofilter                     Graph-Embeddings       Logistic Regression      0.8325   [0.8041, 0.8608]   0.7060
  S_trim_nofilter                     Graph-Embeddings       XGBoost                  0.8305   [0.8079, 0.8531]   0.6939
  S_trim_nofilter                     GRAPE                  XGBoost                  0.8287   [0.8000, 0.8574]   0.6885
  S_trim_nofilter                     LinearRegression       XGBoost                  0.8286   [0.7915, 0.8658]   0.6954
  S_trim_nofilter                     GRAPE                  Logistic Regression      0.8234   [0.7993, 0.8474]   0.6788
  S_trim_nofilter                     GRAPE                  Random Forest            0.8200   [0.79

In [9]:
# Display classification results as a DataFrame
if clf_df is not None:
    display(clf_df)

,Stage,ImpMethod,Classifier,n_seeds,Accuracy_mean,Accuracy_ci_lo,Accuracy_ci_hi,Accuracy_std,F1_Macro_mean,F1_Macro_ci_lo,...,F1_Weighted_ci_hi,F1_Weighted_std,AUROC_mean,AUROC_ci_lo,AUROC_ci_hi,AUROC_std,AUPRC_mean,AUPRC_ci_lo,AUPRC_ci_hi,AUPRC_std
6,S_trim_nofilter,Graph-Embeddings,Logistic Regression,5,0.7029,0.6794,0.7264,0.0169,0.6616,0.6351,...,0.7316,0.0167,0.8325,0.8041,0.8608,0.0204,0.7060,0.6606,0.7514,0.0327
8,S_trim_nofilter,Graph-Embeddings,XGBoost,5,0.7280,0.7124,0.7436,0.0112,0.6241,0.6068,...,0.7096,0.0107,0.8305,0.8079,0.8531,0.0163,0.6939,0.6555,0.7323,0.0276
5,S_trim_nofilter,GRAPE,XGBoost,5,0.7105,0.6790,0.7420,0.0227,0.6149,0.5851,...,0.7107,0.0199,0.8287,0.8000,0.8574,0.0207,0.6885,0.6572,0.7198,0.0225
14,S_trim_nofilter,LinearRegression,XGBoost,5,0.7205,0.6924,0.7486,0.0202,0.6060,0.5678,...,0.7169,0.0226,0.8286,0.7915,0.8658,0.0268,0.6954,0.6522,0.7386,0.0311
3,S_trim_nofilter,GRAPE,Logistic Regression,5,0.6536,0.6320,0.6751,0.0155,0.6222,0.5950,...,0.6936,0.0159,0.8234,0.7993,0.8474,0.0173,0.6788,0.6367,0.7209,0.0303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,S_trim_topk5_rr1_ma1,KNN1,Logistic Regression,5,0.6184,0.5712,0.6656,0.0340,0.5817,0.5308,...,0.6819,0.0336,0.7860,0.7415,0.8306,0.0321,0.6302,0.5727,0.6876,0.0414
471,S_trim_topk5_rr1_ma1,Model,Logistic Regression,5,0.5799,0.5486,0.6112,0.0226,0.5584,0.5304,...,0.6327,0.0262,0.7860,0.7680,0.8039,0.0129,0.6388,0.6197,0.6579,0.0138
465,S_trim_topk5_rr1_ma1,Mean,Logistic Regression,5,0.6059,0.5817,0.6300,0.0174,0.5786,0.5504,...,0.6531,0.0173,0.7801,0.7585,0.8017,0.0156,0.6213,0.5961,0.6465,0.0182
444,S_trim_topk5_rr1_ma1,GAIN,Logistic Regression,5,0.5766,0.5392,0.6140,0.0269,0.5588,0.5270,...,0.6339,0.0227,0.7693,0.7434,0.7952,0.0187,0.6101,0.5798,0.6403,0.0218


## Aggregate Imputation Results

In [ ]:
def aggregate_imputation(results_dir, stages, seeds, output_dir):
    all_dfs = []
    for stage, seed, d in run_dirs(results_dir, stages, seeds):
        path = os.path.join(d, IMP_FILE)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["stage"] = stage
            df["seed"] = seed
            all_dfs.append(df)
        else:
            print(f"  [warn] missing (skip): {path}")

    if not all_dfs:
        print(f"  No {IMP_FILE} found — skipping imputation aggregation.")
        return None

    combined = pd.concat(all_dfs, ignore_index=True)
    group_keys = ["stage", "Modality", "Type", "Method", "Feature"]
    rows = []

    for keys, grp in combined.groupby(group_keys):
        row = dict(zip(group_keys, keys))
        row = {"Stage": row["stage"], "Modality": row["Modality"], "Type": row["Type"],
               "Method": row["Method"], "Feature": row["Feature"],
               "n_seeds": grp["seed"].nunique()}

        if row["Type"] == "categorical":
            for m in ["Accuracy", "AUROC"]:
                if m not in grp.columns:
                    continue
                vals = pd.to_numeric(grp[m], errors="coerce").values
                mean, lo, hi, n = compute_ci(vals)
                row[f"{m}_mean"]  = round(mean, 4) if np.isfinite(mean) else np.nan
                row[f"{m}_ci_lo"] = round(lo,   4) if np.isfinite(lo)   else np.nan
                row[f"{m}_ci_hi"] = round(hi,   4) if np.isfinite(hi)   else np.nan
                row[f"{m}_std"]   = round(float(np.nanstd(vals)), 4)
        else:
            for m in ["MAE", "MSE", "RMSE", "MAPE", "R2"]:
                if m not in grp.columns:
                    continue
                vals = pd.to_numeric(grp[m], errors="coerce").values
                mean, lo, hi, n = compute_ci(vals)
                row[f"{m}_mean"]  = round(mean, 4) if np.isfinite(mean) else np.nan
                row[f"{m}_ci_lo"] = round(lo,   4) if np.isfinite(lo)   else np.nan
                row[f"{m}_ci_hi"] = round(hi,   4) if np.isfinite(hi)   else np.nan
                row[f"{m}_std"]   = round(float(np.nanstd(vals)), 4)
        rows.append(row)

    out_df = pd.DataFrame(rows)

    # Save per-stage files: {stage}_imputation_aggregated.csv
    os.makedirs(output_dir, exist_ok=True)
    for stage in stages:
        stage_df = out_df[out_df["Stage"] == stage]
        if stage_df.empty:
            continue
        out_path = os.path.join(output_dir, f"{stage}_imputation_aggregated.csv")
        stage_df.to_csv(out_path, index=False)
        print(f"  imputation [{stage}] -> {out_path}")

    return out_df


print("Aggregating imputation results...")
imp_df = aggregate_imputation(results_dir, stages, seeds, output_dir)

Aggregating imputation results...


/apps/jupyter/6.5.4/lib/python3.10/site-packages/scipy/stats/_distn_infrastructure.py:2244: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/apps/jupyter/6.5.4/lib/python3.10/site-packages/scipy/stats/_distn_infrastructure.py:2245: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc
/apps/jupyter/6.5.4/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


  imputation [S_trim_nofilter] -> results_ablation_agg/S_trim_nofilter_imputation_aggregated.csv
  imputation [S_trim_topk1] -> results_ablation_agg/S_trim_topk1_imputation_aggregated.csv
  imputation [S_trim_topk3] -> results_ablation_agg/S_trim_topk3_imputation_aggregated.csv
  imputation [S_trim_topk5] -> results_ablation_agg/S_trim_topk5_imputation_aggregated.csv
  imputation [S_trim_topk10] -> results_ablation_agg/S_trim_topk10_imputation_aggregated.csv
  imputation [S_trim_topk20] -> results_ablation_agg/S_trim_topk20_imputation_aggregated.csv
  imputation [S_trim_topk1_gcn1_rr1] -> results_ablation_agg/S_trim_topk1_gcn1_rr1_imputation_aggregated.csv
  imputation [S_trim_topk3_gcn1_rr1] -> results_ablation_agg/S_trim_topk3_gcn1_rr1_imputation_aggregated.csv
  imputation [S_trim_topk5_gcn1_rr1] -> results_ablation_agg/S_trim_topk5_gcn1_rr1_imputation_aggregated.csv
  imputation [S_trim_topk10_gcn1_rr1] -> results_ablation_agg/S_trim_topk10_gcn1_rr1_imputation_aggregated.csv
  impu

In [11]:
# Imputation summary — continuous
if imp_df is not None:
    print("Imputation Summary — Model MAE by Stage/Modality (continuous):")
    print(f"  {'Stage':<35} {'Modality':<8} {'Feature':<20} {'MAE':>8} {'95% CI':>22} {'MAPE':>8}")
    print("  " + "-" * 105)
    model_rows = imp_df[(imp_df["Method"] == "Model") & (imp_df["Type"] == "continuous")]
    for _, r in model_rows.sort_values(["Stage", "Modality", "Feature"]).iterrows():
        ci_str = f"[{r.get('MAE_ci_lo', np.nan):.4f}, {r.get('MAE_ci_hi', np.nan):.4f}]"
        print(f"  {r['Stage']:<35} {r['Modality']:<8} {r['Feature']:<20} "
              f"{r.get('MAE_mean', np.nan):>8.4f} {ci_str:>22} {r.get('MAPE_mean', np.nan):>8.2f}")

    # Categorical summary
    model_cat = imp_df[(imp_df["Method"] == "Model") & (imp_df["Type"] == "categorical")]
    if not model_cat.empty:
        print("\nImputation Summary — Model Accuracy by Stage/Modality (categorical):")
        print(f"  {'Stage':<35} {'Modality':<8} {'Accuracy':>10} {'95% CI':>22} {'AUROC':>8}")
        print("  " + "-" * 87)
        for _, r in model_cat.sort_values(["Stage", "Modality"]).iterrows():
            ci_str = f"[{r.get('Accuracy_ci_lo', np.nan):.4f}, {r.get('Accuracy_ci_hi', np.nan):.4f}]"
            print(f"  {r['Stage']:<35} {r['Modality']:<8} {r.get('Accuracy_mean', np.nan):>10.4f} "
                  f"{ci_str:>22} {r.get('AUROC_mean', np.nan):>8.4f}")

Imputation Summary — Model MAE by Stage/Modality (continuous):
  Stage                               Modality Feature                   MAE                 95% CI     MAPE
  ---------------------------------------------------------------------------------------------------------
  S_trim_nofilter                     M3       BMI                    4.2831       [3.2752, 5.2910]    16.78
  S_trim_nofilter                     M3       DBP                    7.0221       [5.8323, 8.2118]     9.61
  S_trim_nofilter                     M3       Glucose               11.9877     [10.4614, 13.5141]    12.20
  S_trim_nofilter                     M3       HDL                   11.3757      [9.3942, 13.3572]    21.47
  S_trim_nofilter                     M3       HbA1c                  0.4195       [0.3384, 0.5007]     7.16
  S_trim_nofilter                     M3       SBP                   14.1478     [11.7306, 16.5651]    10.44
  S_trim_nofilter                     M3       TotalChol          

In [12]:
# Display imputation results as a DataFrame
if imp_df is not None:
    display(imp_df)

,Stage,Modality,Type,Method,Feature,n_seeds,MAE_mean,MAE_ci_lo,MAE_ci_hi,MAE_std,...,MAPE_ci_hi,MAPE_std,Accuracy_mean,Accuracy_ci_lo,Accuracy_ci_hi,Accuracy_std,AUROC_mean,AUROC_ci_lo,AUROC_ci_hi,AUROC_std
0,S_trim_nofilter,M3,continuous,GAIN,BMI,5,4.3736,2.6898,6.0574,1.2129,...,26.4526,6.8711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S_trim_nofilter,M3,continuous,GAIN,DBP,5,8.9803,4.9926,12.9680,2.8725,...,16.9396,3.7145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S_trim_nofilter,M3,continuous,GAIN,Glucose,5,16.3047,10.8599,21.7496,3.9222,...,22.0733,4.0227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S_trim_nofilter,M3,continuous,GAIN,HDL,5,13.0891,9.9758,16.2024,2.2427,...,27.3219,2.1998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S_trim_nofilter,M3,continuous,GAIN,HbA1c,5,0.4830,0.3896,0.5764,0.0673,...,9.5329,1.0209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4909,S_trim_topk5_rr1_ma1,M7_TRIMMED,continuous,Model,GFAP,5,49.8135,39.0470,60.5800,7.7556,...,63.5516,8.1436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4910,S_trim_topk5_rr1_ma1,M7_TRIMMED,continuous,Model,NFL,5,9.1969,7.0326,11.3612,1.5590,...,88.0723,21.7513,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4911,S_trim_topk5_rr1_ma1,M7_TRIMMED,continuous,Model,Ptau181,5,2.8803,1.1824,4.5781,1.2230,...,68.4170,9.4278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4912,S_trim_topk5_rr1_ma1,M7_TRIMMED,continuous,Model,Ptau217,5,0.2587,0.2195,0.2979,0.0283,...,87.8517,19.6702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
print("Done.")

Done.


## Ablation Summary — Classification

In [14]:
clf_dfs = []
for stage in STAGES:
    for src in ABSTUDY_AGG_SOURCES:
        old_stage = stage
        path = os.path.join(src["base"], f"{old_stage}_classification_aggregated.csv")
        print(path)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["Stage"] = stage   # normalise to canonical name
            clf_dfs.append(df)
            break
    else:
        print(f"  [warn] no clf aggregated CSV for stage={stage}")

if clf_dfs:
    clf_all = pd.concat(clf_dfs, ignore_index=True)
    clf_summary = (
        clf_all[clf_all["ImpMethod"].isin(["Model", "Graph-Embeddings"])]
        .sort_values(["Stage", "ImpMethod", "Classifier"])
        .reset_index(drop=True)
    )
    out_path = os.path.join(output_dir, "abstudy_classification_summary.csv")
    clf_summary.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({len(clf_summary)} rows)")
    display(clf_summary)
else:
    print("No classification files found.")

results_ablation_agg/S_trim_nofilter_classification_aggregated.csv
results_ablation_agg/S_trim_topk1_classification_aggregated.csv
results_ablation_agg/S_trim_topk3_classification_aggregated.csv
results_ablation_agg/S_trim_topk5_classification_aggregated.csv
results_ablation_agg/S_trim_topk10_classification_aggregated.csv
results_ablation_agg/S_trim_topk20_classification_aggregated.csv
results_ablation_agg/S_trim_topk1_gcn1_rr1_classification_aggregated.csv
results_ablation_agg/S_trim_topk3_gcn1_rr1_classification_aggregated.csv
results_ablation_agg/S_trim_topk5_gcn1_rr1_classification_aggregated.csv
results_ablation_agg/S_trim_topk10_gcn1_rr1_classification_aggregated.csv
results_ablation_agg/S_trim_topk20_gcn1_rr1_classification_aggregated.csv
results_ablation_agg/S_trim_nofilter_gcn1_rr1_classification_aggregated.csv
results_ablation_agg/S_trim_topk1_rr1_ma1_classification_aggregated.csv
results_ablation_agg/S_trim_topk3_rr1_ma1_classification_aggregated.csv
  [warn] no clf aggregat

,Stage,ImpMethod,Classifier,n_seeds,Accuracy_mean,Accuracy_ci_lo,Accuracy_ci_hi,Accuracy_std,F1_Macro_mean,F1_Macro_ci_lo,...,F1_Weighted_ci_hi,F1_Weighted_std,AUROC_mean,AUROC_ci_lo,AUROC_ci_hi,AUROC_std,AUPRC_mean,AUPRC_ci_lo,AUPRC_ci_hi,AUPRC_std
0,S_trim_nofilter,Graph-Embeddings,Logistic Regression,5,0.7029,0.6794,0.7264,0.0169,0.6616,0.6351,...,0.7316,0.0167,0.8325,0.8041,0.8608,0.0204,0.7060,0.6606,0.7514,0.0327
1,S_trim_nofilter,Graph-Embeddings,Random Forest,5,0.7096,0.6910,0.7282,0.0134,0.5752,0.5365,...,0.6869,0.0182,0.8028,0.7739,0.8317,0.0208,0.6557,0.6119,0.6996,0.0316
2,S_trim_nofilter,Graph-Embeddings,XGBoost,5,0.7280,0.7124,0.7436,0.0112,0.6241,0.6068,...,0.7096,0.0107,0.8305,0.8079,0.8531,0.0163,0.6939,0.6555,0.7323,0.0276
3,S_trim_nofilter,Model,Logistic Regression,5,0.5967,0.5577,0.6357,0.0281,0.5721,0.5346,...,0.6544,0.0326,0.7835,0.7725,0.7944,0.0079,0.6274,0.6091,0.6457,0.0132
4,S_trim_nofilter,Model,Random Forest,5,0.6837,0.6674,0.7000,0.0117,0.5653,0.5426,...,0.6632,0.0127,0.7827,0.7567,0.8088,0.0188,0.6238,0.5852,0.6624,0.0278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,S_trim_topk5_rr1_ma1,Graph-Embeddings,Random Forest,5,0.7096,0.6857,0.7336,0.0173,0.5770,0.5460,...,0.6867,0.0170,0.8065,0.7767,0.8363,0.0215,0.6527,0.6170,0.6885,0.0257
98,S_trim_topk5_rr1_ma1,Graph-Embeddings,XGBoost,5,0.7205,0.7016,0.7394,0.0136,0.6036,0.5778,...,0.7023,0.0143,0.8277,0.8087,0.8467,0.0137,0.6809,0.6594,0.7024,0.0155
99,S_trim_topk5_rr1_ma1,Model,Logistic Regression,5,0.5799,0.5486,0.6112,0.0226,0.5584,0.5304,...,0.6327,0.0262,0.7860,0.7680,0.8039,0.0129,0.6388,0.6197,0.6579,0.0138
100,S_trim_topk5_rr1_ma1,Model,Random Forest,5,0.6862,0.6635,0.7088,0.0163,0.5778,0.5491,...,0.6755,0.0159,0.7919,0.7599,0.8239,0.0230,0.6317,0.5927,0.6707,0.0281


## Ablation Summary — Imputation (Model, Mean MAPE per Modality, S1–S7)

In [15]:
imp_dfs = []
for stage in STAGES:
    for src in ABSTUDY_AGG_SOURCES:
        old_stage = stage
        path = os.path.join(src["base"], f"{old_stage}_imputation_aggregated.csv")
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["Stage"] = stage   # normalise to canonical name
            imp_dfs.append(df)
            break
    else:
        print(f"  [warn] no imp aggregated CSV for stage={stage}")

if imp_dfs:
    imp_all = pd.concat(imp_dfs, ignore_index=True)

    # Filter: Model method, continuous type, modalities M3/M5/M7
    imp_model = imp_all[
        (imp_all["Method"] == "Model") &
        (imp_all["Type"] == "continuous") &
        (imp_all["Modality"].isin(["M3", "M5", "M7"]))
    ]

    # Mean MAPE_mean per Stage × Modality (averaged over features)
    mape_by_mod = (
        imp_model.groupby(["Stage", "Modality"])["MAPE_mean"]
        .mean()
        .round(4)
        .unstack("Modality")
    )
    mape_by_mod = mape_by_mod.reindex(STAGES)
    mape_by_mod.index.name = "Stage"
    mape_by_mod.columns.name = None

    out_path = os.path.join(output_dir, "abstudy_imputation_summary.csv")
    mape_by_mod.to_csv(out_path)
    print(f"Saved: {out_path}")
    display(mape_by_mod)
else:
    print("No imputation files found.")

Saved: results_ablation_agg/abstudy_imputation_summary.csv


,M3,M5,M7
Stage,,,
S_trim_nofilter,13.6474,45.1657,43.6623
S_trim_topk1,13.5518,44.8063,42.7971
S_trim_topk3,13.5828,45.0136,43.5441
S_trim_topk5,13.4819,44.2991,42.2945
S_trim_topk10,13.5825,45.6126,42.0687
S_trim_topk20,13.7495,45.6258,43.4513
S_trim_topk1_gcn1_rr1,14.8492,44.7375,42.8563
S_trim_topk3_gcn1_rr1,13.6063,45.5829,42.6799
S_trim_topk5_gcn1_rr1,13.7165,45.6571,43.1232
